# TP 2 — Partie 3 : la couche Gold en schéma en étoile, avec dbt

**Objectif :** construire une couche Gold **réutilisable** par toutes les analyses : un schéma en étoile, avec des tables de **faits** (ce qui se mesure) reliées à des tables de **dimensions** (ce qui sert à filtrer et regrouper).

| Table | Type | Grain (une ligne par…) |
|---|---|---|
| `dim_date`, `dim_client`, `dim_produit`, `dim_vendeur` | dimensions | jour, client, produit, vendeur |
| `fait_ventes` | fait | article d'une commande |
| `fait_avis` | fait | avis d'une commande |
| `fait_livraisons` | fait | colis |

Au TP 1, la couche Gold était écrite à la main dans un notebook. Ici, elle est décrite par un **projet dbt** : chaque table est un fichier SQL (un *modèle*), dbt calcule l'ordre d'exécution, lance les **tests de qualité** et génère la documentation.

> **⚠️ dbt démarre sa propre session Spark.** Ce notebook n'ouvre Spark qu'à l'étape 6. Avant de commencer, arrêtez le noyau des autres notebooks (*Shut Down All*).

## Étape 1 — Découvrir le projet

Le projet est dans le dossier `dbt_etoile/`, à côté de ce notebook :

In [ ]:
!find dbt_etoile -name "*.sql" -o -name "*.yml" | grep -v target | sort

| Fichier | Rôle |
|---|---|
| `dbt_project.yml` | nom du projet, connexion utilisée (`lakehouse`), format des tables (Iceberg) |
| `models/sources.yml` | les tables Silver lues par le projet |
| `models/dimensions/*.sql`, `models/faits/*.sql` | un modèle par table Gold : une simple requête `SELECT` |
| `models/schema.yml` | la description des tables et leurs **tests** |
| `tests/*.sql` | des tests écrits en SQL : ils renvoient les lignes en erreur |

Lisez le modèle complet `fait_ventes.sql`. `{{ source('silver', 'orders') }}` désigne la table `silver.orders` : dbt remplace cette expression par le vrai nom de la table, et en déduit que `fait_ventes` dépend de `silver.orders`.

In [ ]:
!cat dbt_etoile/models/faits/fait_ventes.sql

## Étape 2 — Vérifier la connexion

`dbt debug --connection` vérifie que dbt trouve le projet et qu'il se connecte à Spark. Résultat attendu : `Connection test: [OK connection ok]`.

In [ ]:
!dbt debug --connection --project-dir dbt_etoile

## Étape 3 — Compléter les modèles

Ouvrez les fichiers suivants dans JupyterLab (double-clic dans l'explorateur de fichiers), complétez-les en suivant les commentaires, puis enregistrez :

1. `models/dimensions/dim_produit.sql` ;
2. `models/faits/fait_avis.sql` ;
3. `models/faits/fait_livraisons.sql`.

Une **clé de date** est un entier `AAAAMMJJ` : `cast(date_format(une_date, 'yyyyMMdd') as int)`, comme dans `fait_ventes.sql`.

Testez un seul modèle avec l'option `--select` :

In [ ]:
!dbt run --project-dir dbt_etoile --select dim_produit

## Étape 4 — Ajouter les tests

Complétez `models/schema.yml` pour `fait_avis` et `fait_livraisons`, en suivant les commentaires et les exemples des autres tables. Quatre types de tests sont fournis par dbt :

| Test | Vérifie que… |
|---|---|
| `unique` | la colonne n'a pas de doublon |
| `not_null` | la colonne n'a pas de valeur nulle |
| `accepted_values` | la colonne ne prend que les valeurs listées |
| `relationships` | chaque valeur existe dans la colonne d'une autre table (clé étrangère) |

## Étape 5 — Construire et tester toute la couche Gold

`dbt build` exécute tous les modèles **dans l'ordre de leurs dépendances**, et lance les tests de chaque table juste après l'avoir construite. Résultat attendu : `Completed successfully`, aucune erreur.

In [ ]:
!dbt build --project-dir dbt_etoile

> **❓ Question 1.** Dans quel ordre dbt a-t-il construit les tables ? Comment l'a-t-il déterminé, alors qu'aucun fichier n'indique cet ordre ? Que se passe-t-il si un test échoue sur une dimension ?

Générez maintenant la documentation du projet. Ouvrez ensuite le fichier `dbt_etoile/target/static_index.html` : clic droit dans l'explorateur de fichiers > *Open in New Browser Tab*. Cliquez sur l'icône du graphe, en bas à droite, pour voir le *lineage*.

In [ ]:
!dbt docs generate --static --project-dir dbt_etoile

> **❓ Question 2.** Imaginez que la correction de `F.size` ait été oubliée en partie 2 : `nb_incidents` vaudrait `-1` pour les colis de 2017. Quel test l'aurait détecté ? Pourquoi est-il préférable qu'un test échoue plutôt qu'un tableau de bord affiche un nombre d'incidents faux ?

## Étape 6 — Interroger l'étoile

Les tables construites par dbt sont des tables Iceberg ordinaires, dans `lakehouse.gold` : Spark (et, au TP 3, Trino) les lit directement.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("tp2-etoile").getOrCreate()
spark.sql("SHOW TABLES IN lakehouse.gold").filter("tableName LIKE 'dim%' OR tableName LIKE 'fait%'").show()

**Q1.** Les retards de livraison font-ils de mauvais avis ? Comparez la note moyenne et la part d'avis négatifs entre les commandes en retard et les autres.

In [ ]:
# À COMPLÉTER : écrire la requête avec spark.sql(...).show()

**Q2.** Quels sont les 5 états de clients qui donnent la note moyenne la plus basse ? (Jointure fait → dimension : `fait_avis` et `dim_client`.)

In [ ]:
# À COMPLÉTER

**Q3.** Les transporteurs se valent-ils ? Pour chacun : nombre de colis, part de colis avec au moins un incident (colis expédiés à partir de 2018 seulement, puisque les incidents n'existent que dans la version 2 du format) et durée médiane de transit (`percentile_approx(duree_transit_h, 0.5)`).

In [ ]:
# À COMPLÉTER

**Q4.** Un colis qui a subi un incident reçoit-il une moins bonne note ? Reliez les deux faits par la commande (`order_id`).

In [ ]:
# À COMPLÉTER

**Q5.** Attention à la conclusion de Q4 : les incidents touchent surtout les colis **en retard**, qui reçoivent déjà de mauvaises notes. Refaites la comparaison **à retard égal** : ajoutez `en_retard` au regroupement.

In [ ]:
# À COMPLÉTER : même requête que Q4, regroupée par en_retard et par la présence d'un incident

> **❓ Question 3.** Les incidents expliquent-ils vraiment les mauvaises notes ? Que montre la comparaison à retard égal ? Comment s'appelle ce piège ?

> **❓ Question 4 — le piège du grain.** Un collègue calcule le nombre d'avis par catégorie avec la requête ci-dessous. Exécutez-la, puis comparez son total au nombre de lignes de `fait_avis`. D'où vient l'écart ? Corrigez la requête.

In [ ]:
spark.sql("""
    SELECT SUM(nb_avis) AS total_avis FROM (
        SELECT p.categorie, COUNT(*) AS nb_avis
        FROM lakehouse.gold.fait_avis a
        JOIN lakehouse.gold.fait_ventes v ON v.order_id = a.order_id
        JOIN lakehouse.gold.dim_produit p ON p.product_id = v.product_id
        GROUP BY p.categorie)
""").show()
print("lignes de fait_avis :", spark.table("lakehouse.gold.fait_avis").count())

In [ ]:
# À COMPLÉTER : requête corrigée

> **❓ Question 5.** Comparez la couche Gold du TP 1 (une table large, `gold.commandes`) à ce schéma en étoile. Donnez un avantage de chacun.